# Импорт библиотек

In [3]:
import sys
import os

sys.path.append(os.path.abspath('lib'))

from preprocessing_pipeline import create_combined_pipeline
from test_models import run_models_classifications

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Считывание данных

In [5]:
df = pd.read_csv('data/processed.csv')
df.shape

(998, 213)

# Очистка таргета от выбросов

In [7]:
df = df[df['CC50, mM'] < 3500]

# Подготовка данных для эксперемента

In [9]:
combined_pipeline = create_combined_pipeline()
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y1 = df['IC50, mM']
y2 = df['CC50, mM']
y3 = df['SI']
X_transformed = combined_pipeline.fit_transform(X)
df = pd.concat([X_transformed, y1, y2, y3], axis=1)

In [10]:
X = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])
y = df['CC50, mM'].apply(lambda v: 1 if v >= df['CC50, mM'].median() else 0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train dataset size: {X_train.shape}, {y_train.shape}')
print(f'Test dataset size: {X_test.shape}, {y_test.shape}')

Train dataset size: (795, 100), (795,)
Test dataset size: (199, 100), (199,)


# Эксперемент с моделями

In [12]:
run_models_classifications(X_train, X_test, y_train, y_test)

,Model,WA f1,accuracy,WA precision,WA recall,WA support,roc_auc
6,LightGBM,0.76,0.76,0.76,0.76,199.0,0.84
7,CatBoost,0.74,0.74,0.74,0.74,199.0,0.84
2,KNeighbors,0.73,0.73,0.75,0.73,199.0,0.81
3,Random Forest,0.73,0.73,0.74,0.73,199.0,0.84
9,AdaBoost,0.73,0.73,0.73,0.73,199.0,0.81
4,XGBoost,0.72,0.72,0.73,0.72,199.0,0.83
5,Gradient Boosting,0.72,0.72,0.72,0.72,199.0,0.82
8,HistGradientBoosting,0.72,0.72,0.74,0.72,199.0,0.82
1,Decision Tree,0.70,0.70,0.71,0.70,199.0,0.72
0,Logistic Regression,0.65,0.65,0.66,0.65,199.0,0.75


# Подбор гиперпараметров

In [14]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score
from scipy.stats import randint, uniform

param_dist = {
    'iterations': randint(50, 300),
    'depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.3),
    'l2_leaf_reg': uniform(1, 10),
    'border_count': randint(32, 255),
    'grow_policy': ['SymmetricTree', 'Depthwise', 'Lossguide'],
    'random_strength': uniform(0, 1),
    'leaf_estimation_iterations': randint(1, 10),
    'bootstrap_type': ['Bayesian', 'Bernoulli', 'MVS'],
    'feature_border_type': ['GreedyLogSum', 'Median', 'Uniform'],
}
random_search = RandomizedSearchCV(
    estimator=CatBoostClassifier(silent=True, random_state=42),
    param_distributions=param_dist,
    n_iter=200,
    scoring='roc_auc',
    cv=5,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

best_params = random_search.best_params_
print("Лучшие параметры:", best_params)

/Users/v.papadyk/anaconda3/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Лучшие параметры: {'bootstrap_type': 'MVS', 'border_count': 173, 'depth': 9, 'feature_border_type': 'Uniform', 'grow_policy': 'SymmetricTree', 'iterations': 256, 'l2_leaf_reg': 5.275410183585496, 'leaf_estimation_iterations': 3, 'learning_rate': 0.019428755706020276, 'random_strength': 0.6364104112637804}


In [15]:
model = CatBoostClassifier(
    bootstrap_type='MVS',
    border_count=173,
    depth=9,
    feature_border_type='Uniform',
    grow_policy='SymmetricTree',
    iterations=256,
    l2_leaf_reg=5.27,
    leaf_estimation_iterations=3,
    learning_rate=0.02,
    random_strength=0.64,
    random_state=42,
    verbose=0,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

report = classification_report(y_test, y_pred, output_dict=True)
report_df = pd.DataFrame(report).transpose()
roc_auc = roc_auc_score(y_test, y_score)

print("Accuracy:", round(report['accuracy'], 2))
print("ROC AUC Score:", round(roc_auc, 2))
report_df

Accuracy: 0.72
ROC AUC Score: 0.85


,precision,recall,f1-score,support
0,0.787500,0.617647,0.692308,102.000000
1,0.672269,0.824742,0.740741,97.000000
accuracy,0.718593,0.718593,0.718593,0.718593
macro avg,0.729884,0.721195,0.716524,199.000000
weighted avg,0.731332,0.718593,0.715916,199.000000
